First we are goin to load the data, then split the data.

In [1]:
def extract_L_set(filepath):
    l_set = []
    capture = False
    with open(filepath, 'r') as file:
        for line in file:
            line = line.strip()
        
            # Start capturing when we see the header
            if line.startswith("L set:"):
                capture = True
                continue
        
            # Stop capturing when we reach the B vector
            if line.startswith("B vector:"):
                capture = False
                break
            
            # If we are in the capture zone and the line looks like a list
            if capture and line.startswith("["):
                # ast.literal_eval safely converts the string "[1, 2...]" into a Python list
                row = ast.literal_eval(line)
                l_set.append(row)
    return l_set

In [9]:
import numpy as np
import ast #abstract syntax tree module for safe evaluation of string.
loaded_train_test_data = []
#first we loading the L set from 2003 to 2023
filePath = '../ED_Calculation/2003_2023/results/finalize_L_set_from_2003_to_2023.txt'
loaded_train_test_data = np.array(extract_L_set(filePath))


In [10]:
from sklearn.model_selection import train_test_split
#separate the test_data for Feature and Target
X = loaded_train_test_data[:, :5] #first five col of the row is feature
y = loaded_train_test_data[:, 5] # 6th value is the target

#split for train and val
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Shape of X_train {X_train.shape}")
print(f"Shape of y_train{y_train.shape}")
print(f"Shape of X_val{X_val.shape}")
print(f"Shape of y_val{y_val.shape}")

Shape of X_train (116, 5)
Shape of y_train(116,)
Shape of X_val(29, 5)
Shape of y_val(29,)


It is time to define the test data from **2024 to 2025**.

In [17]:
loaded_test_data = []
test_filepath = "../ED_Calculation/2024_current/results/final_result_df_2024_2025.txt"
loaded_test_data = np.array(extract_L_set(test_filepath))
X_test = loaded_test_data[:, :5]
y_test = loaded_test_data[:, 5]


Now we are going to make a Feed Forward Neural Network, which has input layer with 5 feature, a hidden layer with initial neuron size 10 with signmoid function, and a signle neuron output layer with linear function.

In [5]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Input


#Create a model
ffnn = Sequential()
# Input layer (5 features)tells the model to expect 5 columns
ffnn.add(Input(shape=(X_train.shape[1],)))
# Hidden layer (10 neurons) 
ffnn.add(Dense(units=10, activation="sigmoid",))
ffnn.add(Dense(units=1, activation="linear"))
ffnn.summary()


I0000 00:00:1775069795.971511   88974 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1775069796.059577   88974 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1775069798.585538   88974 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
E0000 00:00:1775069799.662209   88974 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 10)             │            60 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            11 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 71 (284.00 B)

 Trainable params: 71 (284.00 B)

 Non-trainable params: 0 (0.00 B)

Now compile and run the model

In [18]:
ffnn.compile(optimizer="adam", loss='mse', metrics=['mae'])
ffnn_history = ffnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_val, y_val), verbose=0)
print("Model compiled and trained successfully.")

Model compiled and trained successfully.


In [20]:
test_results = ffnn.evaluate(X_test, y_test, verbose=0)

In [21]:
print(f"Test Loss: {test_results[1]:.4}")

Test Loss: 0.1901
